### Set up

In [4]:
# imports

from explore_import import  *
import tpp_preprocess as tpp
import hpp_checker as hpp
import data_preprocess as dt

import pyteomics.auxiliary as aux
from pathlib import Path
import os, re, subprocess
warnings.simplefilter(action='ignore', category=FutureWarning)
import ast

In [16]:
# base directories

root="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery"
processed_dir=f"{root}/oui-discovery-vv-data/processed/20251015-oui-discovery-ionbot-results"

In [25]:
# get directory contents
# filter entrapment sub-directories
processed_paths=dt.list_files(processed_dir)

20251015-oui-discovery-ionbot-results/
    PXD005833.v0.11.4/
        openprot-x-trembl-filt-global-outerjoin.csv
        PXD005833-opensearch-x-closedsearch-filt-global.csv.gz
        Metadata_PXD005833.v0.11.4-openprot.txt
        ID-rate-df-filt-hybrid.csv
        ID-rate-df-filt-False.csv
        Metadata_PXD005833.v0.11.4-canon.txt
        Metadata_PXD005833.v0.11.4-trembl.txt
        PXD005833-opensearch-x-closedsearch-filt-global-outerjoin.csv.gz
        openprot-x-trembl-filt-global.csv
        ID-rate-df-filt-custom.csv
        ID-rate-df-filt-global.csv
        ID-rate-df-filt-groupwalk.csv
        PXD005833.v0.11.4-canon/
            PXD005833.v0.11.4-canon-combined-features.csv.gz
            combined-results-w-qvalues.csv.gz
            AM15-canon/
                ionbot.first.proteins.csv
                sample-protein-inference-input.pout
                ionbot.features.csv
                ionbot.first.csv
                group-walk-output.csv
                group-walk-

In [37]:
nonentrapment_paths = processed_paths.copy()
for d, files in processed_paths.items():
    if '-entrap' in d:
        del nonentrapment_paths[d]

In [38]:
len(processed_paths),len(nonentrapment_paths)

(483, 171)

In [39]:
nonentrapment_paths

{'/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/20251015-oui-discovery-ionbot-results/PXD005833.v0.11.4': ['openprot-x-trembl-filt-global-outerjoin.csv',
  'PXD005833-opensearch-x-closedsearch-filt-global.csv.gz',
  'Metadata_PXD005833.v0.11.4-openprot.txt',
  'ID-rate-df-filt-hybrid.csv',
  'ID-rate-df-filt-False.csv',
  'Metadata_PXD005833.v0.11.4-canon.txt',
  'Metadata_PXD005833.v0.11.4-trembl.txt',
  'PXD005833-opensearch-x-closedsearch-filt-global-outerjoin.csv.gz',
  'openprot-x-trembl-filt-global.csv',
  'ID-rate-df-filt-custom.csv',
  'ID-rate-df-filt-global.csv',
  'ID-rate-df-filt-groupwalk.csv'],
 '/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/20251015-oui-discovery-ionbot-results/PXD005833.v0.11.4/PXD005833.v0.11.4-canon': ['PXD005833.v0.11.4-canon-combined-features.csv.gz',
  'combined-results-w-qvalues.csv.gz'],
 '/project/def-marie87/vvshazia/pride_reanalysi

### Indicate global and subset ranks filters on peptide level

In [30]:
#read the files with FDP ranks
root_fdp = '/project/def-marie87/vvshazia/pride_reanalysis/Ionbot_FDP/IonbotData_ionbot_ranks_300925'
#global
global_fdp_file = f'{root_fdp}/results/peptide_level/all_bysample_stats.csv'
global_fdp = pd.read_csv(global_fdp_file)
#correct sample names
global_fdp['sample'] = global_fdp['sample'].str.split('_fdp').str[0]

#subset
subset_fdp_file = f'{root_fdp}/results/peptide_level/all_bysample_stats_subsets.csv'
subset_fdp = pd.read_csv(subset_fdp_file)
#correct sample names
subset_fdp['sample'] = subset_fdp['sample'].str.split('_fdp').str[0].replace('canonical_|noncanonical_', '', regex=True)
#correct subset names
subset_fdp['subset'] = subset_fdp['subset'].map({'canonical':'Canonical','noncanonical':'NonCanonical'})

In [56]:
target = 'group-walk-output-peptide.csv'
qval_score = 'psm_score'
for parent, files in nonentrapment_paths.items():
    for file in files:
        #print(file)
        if file == target:
            print(file)
            database = parent.split('/')[-1].split('-')[-1]
            search_type = 'ClosedSearch' if '-closed' in parent else 'OpenSearch'
            path = os.path.join(parent,file)
            data_peptide = pd.read_csv(path)
            
            #indicate identifications that passed global and subset FDP-rank
            spectrum_file_samp = parent.split('/')[-1].replace('-closed','').replace(f'-{database}','')
            database = 'open' if database == 'openprot' else database            
            #gloabl
            #rank identifications without decoys
            data_peptide['global_rank'] = data_peptide.loc[data_peptide['database'] == 'T', qval_score].rank(method="dense", ascending=False)
            #select sample
            global_fdp_samp = global_fdp[(global_fdp.search_type == search_type)&(global_fdp.database == database)&(global_fdp['sample'] == spectrum_file_samp)]
            if len(global_fdp_samp)==0:
                print(f'Sample NOT FOUND: {database} {spectrum_file_samp}')
                continue #skip ranks assignment entierly
            else:
                #mark targets passing max-rank threshold
                glob_max_rank = global_fdp_samp['Max rank 1% paired FDP'].values[0]
                data_peptide['global_max_rank'] = data_peptide.apply(
                                                                lambda x: x['global_rank'] <= glob_max_rank if x['database'] == 'T' else pd.NA,
                                                                axis=1)
            n_sel = len(data_peptide[data_peptide['global_max_rank']==True])    
            print(f"{database}|{spectrum_file_samp} global max rank and selected IDs: {glob_max_rank} and {round((n_sel/len(data_peptide))*100,4)}%")
            
            #subset
            if database != 'open': 
                #other databases have only one subset 
                data_peptide2 = data_peptide.copy()
                data_peptide2['subset_rank'] = data_peptide2['global_rank']
                data_peptide2['subset_max_rank'] = data_peptide2['global_max_rank']
            else:
                #select sample
                subset_fdp_samp = subset_fdp[(subset_fdp.search_type == search_type)&(subset_fdp['sample'] == spectrum_file_samp)]
                #itter on subsets
                data_peptide2 = pd.DataFrame()
                for subset, sub_data_peptide in data_peptide.groupby('FDRGroup'):
                    if subset == 'Contam': 
                        sub_data_peptide['subset_rank'] = pd.NA
                    else:
                        #mark targets passing max-rank threshold
                        sub_max_rank = subset_fdp_samp[subset_fdp_samp['subset'] == subset]['Max rank 1% paired FDP'].values[0]
                        total_discoveries_fdp = subset_fdp_samp[subset_fdp_samp['subset'] == subset]['Total target discoveries 1% paired FDP'].values[0]
                        
                        sub_data_peptide['subset_rank'] = sub_data_peptide.loc[sub_data_peptide['database'] == 'T', qval_score].rank(method="dense", ascending=False)
                        if total_discoveries_fdp == 0:
                            #non IDs in entrapment - no selection in non-entrapment
                            sub_data_peptide['subset_max_rank'] = sub_data_peptide.apply(
                                                                        lambda x: False if x['database'] == 'T' else pd.NA,
                                                                        axis=1)
                        else:
                            sub_data_peptide['subset_max_rank'] = sub_data_peptide.apply(
                                                                            lambda x: x['subset_rank'] <= sub_max_rank if x['database'] == 'T' else pd.NA,
                                                                            axis=1)
                        n_sel = len(sub_data_peptide[sub_data_peptide['subset_max_rank']==True])    
                        print(f"{database}|{spectrum_file_samp} {subset} subset max rank and selected IDs: {sub_max_rank} and {round((n_sel/len(sub_data_peptide))*100,4)}%")
                    data_peptide2 = pd.concat([data_peptide2, sub_data_peptide])
                
            
            #save
            out = os.path.join(parent, f"{file[:-4]}-vv.csv")
            data_peptide2.to_csv(out,index=False)
#            break
#    break


group-walk-output-peptide.csv
canon|AM15 global max rank and selected IDs: 6357.0 and 44.1873%
group-walk-output-peptide.csv
canon|AM19 global max rank and selected IDs: 7133.0 and 46.7304%
group-walk-output-peptide.csv
canon|AM9 global max rank and selected IDs: 7233.0 and 46.9175%
group-walk-output-peptide.csv
canon|AM14 global max rank and selected IDs: 7507.0 and 47.8665%
group-walk-output-peptide.csv
canon|AM18 global max rank and selected IDs: 5536.0 and 37.362%
group-walk-output-peptide.csv
canon|AM8 global max rank and selected IDs: 5444.0 and 34.4991%
group-walk-output-peptide.csv
canon|AM17 global max rank and selected IDs: 5245.0 and 38.0794%
group-walk-output-peptide.csv
canon|AM7 global max rank and selected IDs: 5133.0 and 44.1048%
group-walk-output-peptide.csv
canon|AM20 global max rank and selected IDs: 5828.0 and 42.9513%
group-walk-output-peptide.csv
canon|AM16 global max rank and selected IDs: 8384.0 and 51.6005%
group-walk-output-peptide.csv
canon|AM21 global max ra

In [57]:
data_peptide2.columns

Index(['ionbot_match_id', 'spectrum_title', 'scan', 'spectrum_file',
       'precursor_mass', 'peptide_mass', 'observed_retention_time', 'charge',
       'database_peptide', 'matched_peptide', 'modifications',
       'unexpected_modification', 'database', 'psm_score', 'proteins',
       'is.decoy', 'q.value', 'leadprot', 'protein_classes', 'isCanonical',
       'isModified', 'custom_q', 'isTarget', 'FDRGroup', 'group_qval',
       'global_rank', 'global_max_rank', 'subset_rank', 'subset_max_rank'],
      dtype='object')